# 02 Model Experiment - Baseline News Classifier

In [1]:
# pandas is used to load and work with CSV data
import pandas as pd 
 
# train_test_split is used to split data into training and testing part
from sklearn.model_selection import train_test_split 

#TfidfVectorizer converts text into numeric features
from sklearn.feature_extraction.text import TfidfVectorizer

# LogisticRegression is the classification model 
from sklearn.linear_model import LogisticRegression 

#accuracy_score checks how many predictions were correct
from sklearn.metrics import accuracy_score


In [2]:
# Load raw kaggle AG News training data
train_path = "../data/raw/train.csv"

df = pd.read_csv(train_path)

#show first 5 rows
df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [4]:
# Rename columns to make them easier to use in python 

df = df.rename(columns={
    "Class Index": "class_index",
    "Title": "title",
    "Description": "description" 
})

df.head()

,class_index,title,description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


# Create label names and text column

In [5]:
# Map numeric labels to readable category names
label_map = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

df["label_name"] = df["class_index"].map(label_map)

In [7]:
# create text
# fill missing values to avoid errors while combining text
df["title"] = df["title"].fillna("")
df["description"] = df["description"].fillna("")

#combine title and description into one text column
df["text"] = df["title"] + "" + df["description"]

In [8]:
# Light cleanup
# Replace weird backslashes and newline with spaces
df["text"] = df["text"].str.replace("\\", "", regex=False)
df["text"] = df["text"].str.replace("\n", "", regex=False)

# Remove spaces from start and end
df["text"] =df["text"].str.strip()

In [9]:
df[["class_index", "label_name", "text"]].head()

,class_index,label_name,text
0,3,Business,Wall St. Bears Claw Back Into the Black (Reute...
1,3,Business,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,3,Business,Iraq Halts Oil Exports from Main Southern Pipe...
4,3,Business,"Oil prices soar to all-time record, posing new..."


# Create balanced small dataset

In [10]:
# Number of rows per class for laptop-friendly training
rows_per_class = 2000

# Take equal number of rows for each class
balanced_df = (
    df.groupby("class_index", group_keys=False)
      .sample(n=rows_per_class, random_state=42)
)

# Shuffle the rows so classes are mixed
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
#check balance:
balanced_df.shape

(8000, 5)

In [13]:
#check balance:
balanced_df["label_name"].value_counts()

label_name
Sports      2000
World       2000
Business    2000
Sci/Tech    2000
Name: count, dtype: int64

# Create X and y

In [14]:
# X means input features (here, X is the news article text)
X = balanced_df["text"]

# y means correct answer/target label (here, y is the class number)
y = balanced_df["class_index"]

In [15]:
#check
X.head()

0    Cardinals to play Broncos in the Liberty BowlM...
1    Martinez targets himself for the most blameNEW...
2    Indonesian police reenact Jakarta embassy blas...
3    Official: Tevez To CorinthiansGoal.com had rev...
4    Steering California #39;s Fight on EmissionsFr...
Name: text, dtype: str

In [16]:
y.head()

0    2
1    2
2    1
3    2
4    3
Name: class_index, dtype: int64

# Train/test split

In [17]:
# split data into training and testing sets
#test_size=0.2 means 20% test, 80% train.
# stratify=y keeps all classes balanced in train and test.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [18]:
# check sizes

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (6400,)
X_test: (1600,)
y_train: (6400,)
y_test: (1600,)


In [19]:
#check class balance in test set
y_test.value_counts()

class_index
4    400
3    400
2    400
1    400
Name: count, dtype: int64

# Convert text to TF-IDF numbers

In [20]:
# TfidfVectorizer converts text into numeric features
#max_features limits vocabulary size to keep training laptop-friendly
vectorizer = TfidfVectorizer(
    max_features=10000,
    lowercase=True,
    stop_words=None
)

In [21]:
# fit on training text
#fit_transform learns vocabulary from trainig data
#and converts training text into numeric matrix
X_train_tfidf = vectorizer.fit_transform(X_train)

In [22]:
# Transform test text

#transform converts test text using the same vocabulary learned from training
X_test_tfidf = vectorizer.transform(X_test)

In [23]:
#check shape

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (6400, 10000)
Test TF-IDF shape: (1600, 10000)


# Train Logistic Regression

In [24]:
# Logistic Regression is a simple and strong baseline classifier for text classification.
model = LogisticRegression(
    max_iter=1000
)

In [25]:
# The model learns patterns between TF-IDF features and category labels
model.fit(X_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

# Predict on test data

In [26]:
# predict categories for test articles
y_pred = model.predict(X_test_tfidf)

In [27]:
print("predictions:", y_pred[:10])
print("Actual:", y_test.iloc[:10].values)

predictions: [4 2 2 2 1 2 2 4 3 3]
Actual: [4 3 2 2 1 2 2 4 3 3]


# Check Accurack

In [28]:
# Accuracy means: correct predictions / total predictions
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.88875


# Predicting my own custom text

In [29]:
id_to_label = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}
#create custom text
custom_text = [
    "Apple announces new AI features for iphone users"
]

In [30]:
# Transform
custom_tfidf = vectorizer.transform(custom_text)

In [31]:
# Predict

custom_prediction = model.predict(custom_tfidf)

predicted_id = custom_prediction[0]
predicted_label = id_to_label[predicted_id]

print("Predicted class ID:", predicted_id)
print("Predicted category:", predicted_label)

Predicted class ID: 4
Predicted category: Sci/Tech


## Day 2 Summary

Today I built a baseline text classifier using a balanced subset of the Kaggle AG News dataset. I created input text and labels, split the data into training and testing sets, converted text into TF-IDF numeric features, trained a Logistic Regression model, checked accuracy, and tested predictions on custom news text.